In [ ]:
import os
import requests
from tqdm import tqdm

url = "https://data.ceda.ac.uk/neodc/esacci/fire/data/burned_area/Sentinel3_SYN/pixel/v1.1/2024/20240601-ESACCI-L3S_FIRE-BA-SYN-AREA_4-fv1.1.tif"
out_path = "firecci_20240601.tif"

if not os.path.exists(out_path):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in tqdm(r.iter_content(chunk_size=8192)):
            f.write(chunk)

print("Downloaded:", out_path)

In [ ]:
import rasterio
import numpy as np

with rasterio.open("firecci_20240601.tif") as src:
    firecci = src.read(1)
    firecci_meta = src.meta

print("Shape:", firecci.shape)
print("Min / Max:", firecci.min(), firecci.max())

In [ ]:
gt_mask = (firecci > 0).astype(np.uint8)

print("Burned pixels:", gt_mask.sum())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.imshow(gt_mask, cmap="hot")
plt.title("FireCCI L3 Burned Area (Binary GT)")
plt.axis("off")
plt.show()

In [ ]:
from pyproj import Transformer

lat, lon = 39.12, 27.18  # FIRMS örnek nokta

with rasterio.open("firecci_20240601.tif") as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    row, col = src.index(x, y)
    val = src.read(1)[row, col]

print("FireCCI value at FIRMS point:", val)
print("Burned?", val > 0)

In [ ]:
meta = firecci_meta.copy()
meta.update(dtype=rasterio.uint8, count=1)

with rasterio.open("firecci_gt_binary.tif", "w", **meta) as dst:
    dst.write(gt_mask, 1)

print("Saved GT mask: firecci_gt_binary.tif")